# 12장 실습 ① — 어텐션과 위치 인코딩

**TensorFlow 판**

11장의 「어려운 규칙」을 어텐션으로 풉니다. 그리고 **위치 인코딩를
빼면 무슨 일이 생기는지** 봅니다.

> **어텐션은 가중 합입니다. 그리고 합은 순서를 따지지 않습니다.**
> 11장 §11.7의 「임베딩 + 평균」이 여기서 다시 나옵니다.

## 12.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 12.1 실험대 — 11장의 어려운 규칙

**Conv1D가 0.949에 머물고 LSTM만 1.000을 냈던** 그 과제입니다.
어텐션이 어떻게 하는지 봅니다.

In [ ]:
# 11장 §11.9의 「어려운 규칙」을 그대로 씁니다.
# 감성 단어가 둘(부호 반대), 정답은 **먼저 나온 쪽**을 따릅니다.
V = len(data.TOY_VOCAB)
LENGTH = 16
x, y = data.toy_reviews(n=8000, length=LENGTH, seed=42, hard=True)
sp = data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=42)
print(sp.summary())
print()
for k in range(3):
    print(f"  {data.toy_decode(x[k]):<52} → {'긍정' if y[k] else '부정'}")
print()
print("이 과제는 **떨어져 있는 두 자리를 견주어야** 풀립니다.")
print("11장에서 Conv1D가 0.949에 머물고 LSTM만 1.000을 냈던 그 과제입니다.")

## 12.2 위치 인코딩

어텐션에는 **순서 개념이 없습니다.** 그래서 순서를 **입력에 심어** 줍니다.

In [ ]:
# 위치 인코딩 — 사인·코사인. (Vaswani et al. 2017)
# 순수 numpy입니다. 세 판이 **완전히 같습니다.**
def positional_encoding(length, depth):
    """자리마다 다른 값을 갖는 (length, depth) 행렬을 만든다.

    같은 자리는 늘 같은 값이고, 가까운 자리끼리는 비슷한 값이 된다.
    이것을 임베딩에 **더해** 주면 "몇 번째 단어인가"가 표현에 들어간다.
    """
    pos = np.arange(length)[:, None]
    i = np.arange(depth)[None, :]
    angle = pos / np.power(10000.0, (2 * (i // 2)) / depth)
    pe = np.zeros((length, depth), dtype="float32")
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

PE = positional_encoding(LENGTH, 8)

fig, ax = plt.subplots(figsize=(7.0, 3.2))
im = ax.imshow(PE.T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("자리 (0~15)"); ax.set_ylabel("차원 (0~7)")
ax.set_title("위치 인코딩 — 자리마다 다른 무늬")
fig.colorbar(im, ax=ax, shrink=0.85); plt.tight_layout(); plt.show()

print("→ 세로줄 하나가 자리 하나입니다. **자리마다 무늬가 다릅니다.**")
print("→ 이것을 임베딩에 더하면 같은 단어라도 자리가 다르면 다른 벡터가 됩니다.")

## 12.3 모델 정의 — 여기만 판마다 다릅니다

**PyTorch 판의 `batch_first=True`** 에 주목하십시오. 이것을 빼면
`nn.MultiheadAttention` 은 (길이, 배치, 차원) 순서를 기대합니다.
**오류 없이 조용히 틀린 결과가 나오는** 자리입니다.

In [ ]:
import tensorflow as tf

L_ = tf.keras.layers

class AddPositional(L_.Layer):
    """위치 인코딩를 더하는 층. 학습되는 파라미터가 없습니다."""
    def __init__(self, length, depth, **kw):
        super().__init__(**kw)
        self.pe = tf.constant(positional_encoding(length, depth))

    def call(self, x):
        return x + self.pe

def _build(kind, V, L, d=8):
    """모델 정의 — **이 함수만 판마다 다릅니다.**"""
    inp = L_.Input(shape=(L,))
    x = L_.Embedding(V, d)(inp)
    if kind == "attn_pos":
        x = AddPositional(L, d)(x)
    if kind in ("attn", "attn_pos"):
        a = L_.MultiHeadAttention(num_heads=2, key_dim=d // 2)(x, x)
        x = L_.LayerNormalization()(L_.Add()([x, a]))
        x = L_.GlobalAveragePooling1D()(x)
    elif kind == "cnn":
        x = L_.GlobalMaxPooling1D()(L_.Conv1D(16, 3, activation="relu")(x))
    elif kind == "lstm":
        x = L_.LSTM(32)(x)
    return tf.keras.Model(inp, L_.Dense(1, activation="sigmoid")(x))

_FIT = {}

def train_text(kind, sp, lr=0.003, seed=42, epochs=25):
    """(시험 정확도, 파라미터 수, 모델) 을 돌려준다."""
    dlbook.set_seed(seed)
    V, L = len(data.TOY_VOCAB), sp.x_train.shape[1]
    m = _build(kind, V, L)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr, clipnorm=1.0),
              loss="binary_crossentropy")
    m.fit(sp.x_train, sp.y_train, epochs=dlbook.smoke.epochs(epochs),
          batch_size=64, verbose=0)
    _FIT[kind] = m
    pred = (m.predict(sp.x_test, verbose=0).reshape(-1) > 0.5).astype(int)
    return metrics.accuracy(sp.y_test, pred), m.count_params(), m

def predict_acc(kind, sp, xs):
    m = _FIT[kind]
    return metrics.accuracy(
        sp.y_test, (m.predict(xs, verbose=0).reshape(-1) > 0.5).astype(int))

def attention_weights(model, xs):
    """학습된 모델에서 어텐션 가중치를 꺼낸다. (머리, 질의, 키)"""
    emb = [l for l in model.layers if isinstance(l, L_.Embedding)][0]
    pos = [l for l in model.layers if isinstance(l, AddPositional)]
    mha = [l for l in model.layers if isinstance(l, L_.MultiHeadAttention)][0]
    h = emb(xs)
    if pos:
        h = pos[0](h)
    _, scores = mha(h, h, return_attention_scores=True)
    return np.asarray(scores)[0]

## 12.4 네 모델을 나란히

In [ ]:
kinds = [("cnn",      "임베딩 + Conv1D"),
         ("lstm",     "임베딩 + LSTM"),
         ("attn",     "어텐션 (위치 인코딩 없음)"),
         ("attn_pos", "어텐션 + 위치 인코딩")]

print(f"{'모델':<28}{'파라미터':>12}{'시험 정확도':>14}")
print("-" * 54)
for kind, ko in kinds:
    acc, n_params, _ = train_text(kind, sp)
    print(f"{ko:<28}{n_params:>12,}{acc:>14.3f}")
    dlbook.record(f"ch12_{kind}_acc", acc)
    dlbook.record(f"ch12_{kind}_params", n_params)

print()
print("★ 위치 인코딩가 없는 어텐션은 **동전 던지기**입니다.")
print("  11장 §11.7의 「임베딩 + 평균」과 똑같은 이유입니다 —")
print("  어텐션은 가중 합이고, **합은 순서를 따지지 않습니다.**")
print()
print("★ 위치 인코딩를 더하면 **LSTM과 같은 성능을 파라미터 1/7로** 냅니다.")

## 12.5 확인 — 위치 인코딩가 정말 순서를 넣는가

11장 §11.8에서 쓴 것과 같은 검사입니다. **시험 문장의 단어를 섞습니다.**

In [ ]:
# 확인 — 위치 인코딩가 정말 순서를 넣어 주는가.
rng = np.random.default_rng(0)
x_shuf = np.stack([rng.permutation(r) for r in sp.x_test])

print(f"{'모델':<12}{'원래 순서':>12}{'섞은 뒤':>12}{'떨어진 폭':>12}")
print("-" * 48)
for kind in ("attn", "attn_pos", "lstm"):
    train_text(kind, sp)
    a_ord = predict_acc(kind, sp, sp.x_test)
    a_shuf = predict_acc(kind, sp, x_shuf)
    print(f"{kind:<12}{a_ord:>12.3f}{a_shuf:>12.3f}{a_ord - a_shuf:>12.3f}")
    dlbook.record(f"ch12_shuffle_drop_{kind}", float(a_ord - a_shuf))

print()
print("→ 위치 인코딩가 **없는** 어텐션은 섞어도 그대로(0.000)입니다.")
print("   순서를 애초에 보지 않았습니다.")
print("→ 위치 인코딩를 더한 어텐션은 무너집니다. **순서를 보고 있었습니다.**")

## 정리

| 모델 | 파라미터 | 정확도 |
|---|:--:|:--:|
| 임베딩 + Conv1D | 633 | 0.949 |
| 임베딩 + LSTM | 5,497 | 1.000 |
| 어텐션 (위치 인코딩 없음) | 809 | **0.494** |
| **어텐션 + 위치 인코딩** | **809** | **0.996** |

- **어텐션만으로는 순서를 못 봅니다.** 가중 합이고, 합은 순서를 따지지
  않습니다. 11장 §11.7의 「임베딩 + 평균」과 정확히 같은 이유입니다.
- **위치 인코딩를 더하면 LSTM과 같은 성능을 파라미터 1/7로** 냅니다.
- 위치 인코딩에는 **학습되는 파라미터가 없습니다.** 사인·코사인 값을
  더할 뿐인데 0.494가 0.996이 됩니다.

### 연습

1. 위치 인코딩를 **학습되는 임베딩**(`Embedding(LENGTH, 8)`)으로 바꾸십시오.
   성능이 달라집니까. 파라미터는 몇 개 늘어납니까.
2. 머리 수(`num_heads`)를 1, 4로 바꾸십시오.
3. 잔차 연결(`Add`)과 정규화(`LayerNormalization`)를 빼면 어떻게 됩니까.
4. 위치 인코딩를 **곱하기**로 바꾸면 어떻게 됩니까. 왜 더하기입니까.